# Knapsack
## Dati
Abbiamo 
- Uno **zaino** con capacità $W$
- Un insieme di **oggetti** $N =\left\{1, \dots, j, \dots, n\right\}$ 
> $N =\left\{\text{borraccia}, \text{penna}, \text{schiscetta} \right\}$ 
- Ogni **oggetto** ha 
    - Un **peso** $w_j >0 , \quad j \in N $
    - Un **profitto** $p_j>0 ,\quad  j \in N$
- Un **vincolo** di peso $w_{max}$

| **Oggetto**   | **Peso**  | **Profitto**  |
| :---:         | :---:     | :---:         |
| Calzini       |   7       |   7           |
| Pantaloni     |   4       |   6           |
| Fiammiferi    |   1       |   2           |    
| Spazzolino    |   4       |   10          |
| Mattone       |   10      |   1           |

## Obiettivo
Scegliere un insieme di $S \subseteq   N $ oggetti di modo da **massimizzare** il profitto

In [1]:
import pyomo.environ as pyo

oggetti = [
    'Calzini', 
    'Pantaloni', 
    'Fiammiferi', 
    'Spazzolino', 
    'Mattone']
pesi = [
    7,
    4,
    1,
    4,
    10 ]
profitti = [
    7,
    6,
    2,
    10,
    1 ]
# Capacità
w_max=14

w=dict(zip(oggetti, pesi))
print(w)
p=dict(zip(oggetti, profitti))
print(p)

#Creo il modello
model= pyo.ConcreteModel(name="Knapsack")

#Creo le variabili nel modello
model.x = pyo.Var(oggetti, within=pyo.Binary)

#Creo l'obiettivo nel modello
def funzione_obiettivo(m: pyo.ConcreteModel) -> pyo.Expression:
    """
    Calcola il profitto totale: sommatoria di (profitto_i * variabile_i)
    per ogni oggetto i nel set degli oggetti.
    """
    return sum(p[j]*m.x[j] for j in oggetti)

model.obiettivo = pyo.Objective(rule = funzione_obiettivo, sense=pyo.maximize)

def vincoli_modello(m: pyo.ConcreteModel) -> pyo.Expression:
    """
    Valuta la condizione se il peso totale degli oggetti è inferiore alla capacità : sommatoria di (peso_i * variabile_i)
    """
    return sum(w[j]*m.x[j] for j in oggetti) <= w_max

model.vincoli = pyo.Constraint(rule= vincoli_modello)

ottimizzatore = pyo.SolverFactory('glpk')

#Se metti tee= True stampa tutta l'esecuzione
risultato=ottimizzatore.solve(model, tee=False)

model.display()

{'Calzini': 7, 'Pantaloni': 4, 'Fiammiferi': 1, 'Spazzolino': 4, 'Mattone': 10}
{'Calzini': 7, 'Pantaloni': 6, 'Fiammiferi': 2, 'Spazzolino': 10, 'Mattone': 1}
Model Knapsack

  Variables:
    x : Size=5, Index={Calzini, Pantaloni, Fiammiferi, Spazzolino, Mattone}
        Key        : Lower : Value : Upper : Fixed : Stale : Domain
           Calzini :     0 :   1.0 :     1 : False : False : Binary
        Fiammiferi :     0 :   1.0 :     1 : False : False : Binary
           Mattone :     0 :   0.0 :     1 : False : False : Binary
         Pantaloni :     0 :   0.0 :     1 : False : False : Binary
        Spazzolino :     0 :   1.0 :     1 : False : False : Binary

  Objectives:
    obiettivo : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True :  19.0

  Constraints:
    vincoli : Size=1
        Key  : Lower : Body : Upper
        None :  None : 12.0 :  14.0
